### 2 - Teste Cálculos Indicadores

#### Importação dos dados tratados

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Dimensão original df:", df.shape)

Dimensão original df: (10445, 18)


In [2]:
df.head()

,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
3,2017,Em curso,Branca,66262151,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
4,2017,Evadidos,Branca,66262139,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2017-05-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Transf. externa,Elétrica,Técnico-Integrado,Matutino


In [3]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

Testando alguns agrupamentos:

In [4]:
contagem_matriculas_ano_curso = df.groupby(["Ano", "Nome de Curso"])["Código da Matricula"].count().reset_index() #resetamos o index para permitir agrupar/salvar em excel/csv
print(contagem_matriculas_ano_curso)

     Ano                          Nome de Curso  Código da Matricula
0   2017  Análise e Desenvolvimento de Sistemas                  247
1   2017                  Eletrônica Industrial                   84
2   2017           Gestão Desportiva e de Lazer                  112
3   2017          Letras - Português e Espanhol                   30
4   2017                    Técnico em Comércio                   33
..   ...                                    ...                  ...
81  2024                    Técnico em Comércio                  127
82  2024                  Técnico em Eletrônica                  164
83  2024             Técnico em Guia de Turismo                   77
84  2024                 Técnico em Informática                  217
85  2024                       Técnico em Lazer                  117

[86 rows x 3 columns]


#### Indicadores por Curso e Ano:

In [5]:
df["Concluinte_Previsto"] = (
    df["Data de Fim Previsto do Ciclo"].dt.year == df["Ano"]
)

df_indicadores = (
    df.groupby(["Ano", "Nome de Curso"])
      .agg(
          matriculas_atendidas = ("Código da Matricula", "nunique"),
          matriculas_ativas_regulares = ("Situação de Matrícula", lambda x: (x == "Em fluxo").sum()),
          matriculas_ativas_retidas = ("Situação de Matrícula", lambda x: (x == "Retido").sum()),
          concluintes_no_prazo= ("Situação de Matrícula", lambda x: (x == "Concluída no prazo").sum()),
          concluintes_com_atraso= ("Situação de Matrícula", lambda x: (x == "Concluída com atraso").sum()),
          abandono = ("Situação de Matrícula", lambda x: (x == "Abandono").sum()),
          desligado = ("Situação de Matrícula", lambda x: (x == "Desligada").sum()),
          transferencia_externa = ("Situação de Matrícula", lambda x: (x == "Transf. externa").sum()),
          transferencia_interna = ("Situação de Matrícula", lambda x: (x == "Transf. interna").sum()),
          integralizada = ("Situação de Matrícula", lambda x: (x == "Integralizada").sum()),
          concluintes_previstos=("Concluinte_Previsto", "sum"),
        )
      .reset_index()
)

df_indicadores["taxa_conclusao"] = (
        ((df_indicadores["concluintes_no_prazo"] + df_indicadores["concluintes_com_atraso"] + df_indicadores["integralizada"]) / df_indicadores["matriculas_atendidas"] )* 100
    )
df_indicadores["taxa_evasao"] = (
        ((df_indicadores["abandono"] + df_indicadores["desligado"] + df_indicadores["transferencia_externa"]) / df_indicadores["matriculas_atendidas"]) * 100
    )
df_indicadores["taxa_retencao"] = (
        (df_indicadores["matriculas_ativas_retidas"] / df_indicadores["matriculas_atendidas"]) * 100
    )
df_indicadores["indice_eficiencia"] = (
    (df_indicadores["concluintes_no_prazo"] / (df_indicadores["concluintes_no_prazo"] + 
                                                                              df_indicadores["concluintes_com_atraso"] +
                                                                              df_indicadores["abandono"] + df_indicadores["desligado"] +
                                                                              df_indicadores["transferencia_interna"]+
                                                                              df_indicadores["transferencia_externa"])) * 100
    )
    
df_indicadores["taxa_matr_cont_regular"] = (
        (df_indicadores["matriculas_ativas_regulares"]/df_indicadores["matriculas_atendidas"]) * 100
    )
    
df_indicadores["taxa_permanencia_exito"] = (
  df_indicadores["taxa_conclusao"] + df_indicadores["taxa_matr_cont_regular"]
    )
    
df_indicadores["taxa_efetividade_academica"] = (
  (df_indicadores["concluintes_no_prazo"]/df_indicadores["concluintes_previstos"]) * 100
    )
    
colunas_taxas = [
    "taxa_conclusao",
    "taxa_evasao",
    "taxa_retencao",
    "indice_eficiencia",
    "taxa_matr_cont_regular",
    "taxa_permanencia_exito",
    "taxa_efetividade_academica"
]

df_indicadores[colunas_taxas] = df_indicadores[colunas_taxas].round(2)    
    

In [6]:
df_indicadores.head()

,Ano,Nome de Curso,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes_no_prazo,concluintes_com_atraso,abandono,desligado,transferencia_externa,transferencia_interna,integralizada,concluintes_previstos,taxa_conclusao,taxa_evasao,taxa_retencao,indice_eficiencia,taxa_matr_cont_regular,taxa_permanencia_exito,taxa_efetividade_academica
0,2017,Análise e Desenvolvimento de Sistemas,247,163,31,5,12,25,8,2,1,0,27,6.88,14.17,12.55,9.43,65.99,72.87,18.52
1,2017,Eletrônica Industrial,84,59,0,0,0,19,6,0,0,0,11,0.00,29.76,0.00,0.00,70.24,70.24,0.00
2,2017,Gestão Desportiva e de Lazer,112,74,6,8,4,13,5,0,0,2,20,12.50,16.07,5.36,26.67,66.07,78.57,40.00
3,2017,Letras - Português e Espanhol,30,25,0,0,0,0,5,0,0,0,0,0.00,16.67,0.00,0.00,83.33,83.33,NaN
4,2017,Técnico em Comércio,33,33,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,NaN,100.00,100.00,NaN


In [7]:
df_indicadores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Ano                          86 non-null     int64  
 1   Nome de Curso                86 non-null     object 
 2   matriculas_atendidas         86 non-null     int64  
 3   matriculas_ativas_regulares  86 non-null     int64  
 4   matriculas_ativas_retidas    86 non-null     int64  
 5   concluintes_no_prazo         86 non-null     int64  
 6   concluintes_com_atraso       86 non-null     int64  
 7   abandono                     86 non-null     int64  
 8   desligado                    86 non-null     int64  
 9   transferencia_externa        86 non-null     int64  
 10  transferencia_interna        86 non-null     int64  
 11  integralizada                86 non-null     int64  
 12  concluintes_previstos        86 non-null     int64  
 13  taxa_conclusao        

#### Tratamento dos valores ausentes:

In [8]:
df_indicadores[df_indicadores["indice_eficiencia"].isna()]

,Ano,Nome de Curso,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes_no_prazo,concluintes_com_atraso,abandono,desligado,transferencia_externa,transferencia_interna,integralizada,concluintes_previstos,taxa_conclusao,taxa_evasao,taxa_retencao,indice_eficiencia,taxa_matr_cont_regular,taxa_permanencia_exito,taxa_efetividade_academica
4,2017,Técnico em Comércio,33,33,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,NaN,100.0,100.0,NaN
13,2018,Processos Gerenciais,40,40,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,NaN,100.0,100.0,NaN
14,2018,Técnico em Agroecologia,61,61,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,NaN,100.0,100.0,NaN
37,2020,Técnico em Comércio,113,93,20,0,0,0,0,0,0,0,28,0.0,0.0,17.7,NaN,82.3,82.3,0.0


In [9]:
df_indicadores[df_indicadores["taxa_efetividade_academica"].isna()]

,Ano,Nome de Curso,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes_no_prazo,concluintes_com_atraso,abandono,desligado,transferencia_externa,transferencia_interna,integralizada,concluintes_previstos,taxa_conclusao,taxa_evasao,taxa_retencao,indice_eficiencia,taxa_matr_cont_regular,taxa_permanencia_exito,taxa_efetividade_academica
3,2017,Letras - Português e Espanhol,30,25,0,0,0,0,5,0,0,0,0,0.00,16.67,0.00,0.0,83.33,83.33,NaN
4,2017,Técnico em Comércio,33,33,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,NaN,100.00,100.00,NaN
7,2017,Técnico em Informática,48,46,0,0,0,0,0,2,0,0,0,0.00,4.17,0.00,0.0,95.83,95.83,NaN
8,2017,Técnico em Lazer,55,51,0,0,0,0,1,3,0,0,0,0.00,7.27,0.00,0.0,92.73,92.73,NaN
12,2018,Letras - Português e Espanhol,58,51,0,0,0,0,4,3,0,0,0,0.00,12.07,0.00,0.0,87.93,87.93,NaN
13,2018,Processos Gerenciais,40,40,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,NaN,100.00,100.00,NaN
14,2018,Técnico em Agroecologia,61,61,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,NaN,100.00,100.00,NaN
15,2018,Técnico em Comércio,63,61,0,0,0,0,2,0,0,0,0,0.00,3.17,0.00,0.0,96.83,96.83,NaN
18,2018,Técnico em Informática,79,73,0,0,0,0,0,5,1,0,0,0.00,6.33,0.00,0.0,92.41,92.41,NaN
23,2019,Letras - Português e Espanhol,89,75,0,0,0,6,8,0,0,0,0,0.00,15.73,0.00,0.0,84.27,84.27,NaN


In [10]:
# os NaN foram resultantes de divisão por zero, então serão substituídos por zero:
df_indicadores = df_indicadores.fillna({"indice_eficiencia": 0, "taxa_efetividade_academica": 0})

In [11]:
df_indicadores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Ano                          86 non-null     int64  
 1   Nome de Curso                86 non-null     object 
 2   matriculas_atendidas         86 non-null     int64  
 3   matriculas_ativas_regulares  86 non-null     int64  
 4   matriculas_ativas_retidas    86 non-null     int64  
 5   concluintes_no_prazo         86 non-null     int64  
 6   concluintes_com_atraso       86 non-null     int64  
 7   abandono                     86 non-null     int64  
 8   desligado                    86 non-null     int64  
 9   transferencia_externa        86 non-null     int64  
 10  transferencia_interna        86 non-null     int64  
 11  integralizada                86 non-null     int64  
 12  concluintes_previstos        86 non-null     int64  
 13  taxa_conclusao        

testando outra forma de calcular os indicadores:

In [12]:
def calcular_indicadores(df, agrupamento):
    indicadores = (
        df.groupby(agrupamento)
          .agg(
            matriculas_atendidas = ("Código da Matricula", "nunique"),
            matriculas_ativas_regulares = ("Situação de Matrícula", lambda x: (x == "Em fluxo").sum()),
            matriculas_ativas_retidas = ("Situação de Matrícula", lambda x: (x == "Retido").sum()),
            concluintes_no_prazo = ("Situação de Matrícula", lambda x: (x == "Concluída no prazo").sum()),
            concluintes_com_atraso = ("Situação de Matrícula", lambda x: (x == "Concluída com atraso").sum()),
            abandono = ("Situação de Matrícula", lambda x: (x == "Abandono").sum()),
            desligados = ("Situação de Matrícula", lambda x: (x == "Desligada").sum()),
            transferencia_interna = ("Situação de Matrícula", lambda x: (x == "Transf. externa").sum()),
            transferencia_externa = ("Situação de Matrícula", lambda x: (x == "Transf. interna").sum()),
            integralizadas = ("Situação de Matrícula", lambda x: (x == "Integralizada").sum()),
            concluintes_previstos=("Concluinte_Previsto", "sum"),
          )
          .reset_index()
    )

    indicadores["taxa_conclusao"] = (
        (indicadores["concluintes_no_prazo"] + indicadores["concluintes_com_atraso"]+ indicadores["integralizadas"]) / indicadores["matriculas_atendidas"] * 100
    )
    indicadores["taxa_evasao"] = (
        (indicadores["abandono"] + indicadores["desligados"] + indicadores["transferencia_externa"]) / indicadores["matriculas_atendidas"] * 100
    )
    indicadores["taxa_retencao"] = (
        indicadores["matriculas_ativas_retidas"] / indicadores["matriculas_atendidas"] * 100
    )
    indicadores["indice_eficiencia"] = ( indicadores["concluintes_no_prazo"] / (indicadores["concluintes_no_prazo"] + 
                                                                              indicadores["concluintes_com_atraso"] +
                                                                              indicadores["abandono"] + indicadores["desligados"] +
                                                                              indicadores["transferencia_interna"]+
                                                                              indicadores["transferencia_externa"]) * 100
    )
    
    indicadores["taxa_matr_cont_regular"] = (
        indicadores["matriculas_ativas_regulares"]/indicadores["matriculas_atendidas"] * 100
    )
    
    indicadores["taxa_permanencia_exito"] = indicadores["taxa_conclusao"] + indicadores["taxa_matr_cont_regular"]
    
    indicadores["taxa_efetividade_academica"] = indicadores["concluintes_no_prazo"]/indicadores["concluintes_previstos"]
    
    indicadores = indicadores.fillna(0)

    return indicadores.round(2)

In [13]:
df_indicadores_ano = calcular_indicadores(df, ["Ano"])
df_indicadores_ano.head(20)

,Ano,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes_no_prazo,concluintes_com_atraso,abandono,desligados,transferencia_interna,transferencia_externa,integralizadas,concluintes_previstos,taxa_conclusao,taxa_evasao,taxa_retencao,indice_eficiencia,taxa_matr_cont_regular,taxa_permanencia_exito,taxa_efetividade_academica
0,2017,811,569,42,48,20,70,45,13,2,2,105,8.63,14.43,5.18,24.24,70.16,78.79,0.46
1,2018,1026,765,79,67,13,40,46,15,1,0,155,7.80,8.48,7.70,36.81,74.56,82.36,0.43
2,2019,1285,991,73,61,17,103,33,6,0,1,227,6.15,10.58,5.68,27.73,77.12,83.27,0.27
3,2020,1223,960,166,0,2,78,10,4,3,0,285,0.16,7.44,13.57,0.00,78.50,78.66,0.00
4,2021,1122,762,268,0,28,27,18,19,0,0,319,2.50,4.01,23.89,0.00,67.91,70.41,0.00
5,2022,1497,728,518,50,62,83,36,20,0,0,317,7.48,7.95,34.60,19.92,48.63,56.11,0.16
6,2023,1669,885,533,97,66,47,23,18,0,0,203,9.77,4.19,31.94,38.65,53.03,62.79,0.48
7,2024,1812,1024,637,56,51,1,33,8,0,2,212,6.02,1.88,35.15,37.58,56.51,62.53,0.26


In [14]:
df_indicadores_curso_ano = calcular_indicadores(df, ["Ano", "Nome de Curso"])
df_indicadores_curso_ano.head(10)

,Ano,Nome de Curso,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes_no_prazo,concluintes_com_atraso,abandono,desligados,transferencia_interna,transferencia_externa,integralizadas,concluintes_previstos,taxa_conclusao,taxa_evasao,taxa_retencao,indice_eficiencia,taxa_matr_cont_regular,taxa_permanencia_exito,taxa_efetividade_academica
0,2017,Análise e Desenvolvimento de Sistemas,247,163,31,5,12,25,8,2,1,0,27,6.88,13.77,12.55,9.43,65.99,72.87,0.19
1,2017,Eletrônica Industrial,84,59,0,0,0,19,6,0,0,0,11,0.00,29.76,0.00,0.00,70.24,70.24,0.00
2,2017,Gestão Desportiva e de Lazer,112,74,6,8,4,13,5,0,0,2,20,12.50,16.07,5.36,26.67,66.07,78.57,0.40
3,2017,Letras - Português e Espanhol,30,25,0,0,0,0,5,0,0,0,0,0.00,16.67,0.00,0.00,83.33,83.33,0.00
4,2017,Técnico em Comércio,33,33,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
5,2017,Técnico em Eletrônica,115,76,4,11,3,4,10,6,1,0,23,12.17,13.04,3.48,31.43,66.09,78.26,0.48
6,2017,Técnico em Guia de Turismo,87,42,1,24,1,9,10,0,0,0,24,28.74,21.84,1.15,54.55,48.28,77.01,1.00
7,2017,Técnico em Informática,48,46,0,0,0,0,0,2,0,0,0,0.00,0.00,0.00,0.00,95.83,95.83,0.00
8,2017,Técnico em Lazer,55,51,0,0,0,0,1,3,0,0,0,0.00,1.82,0.00,0.00,92.73,92.73,0.00
9,2018,Análise e Desenvolvimento de Sistemas,267,173,54,10,4,15,11,0,0,0,46,5.24,9.74,20.22,25.00,64.79,70.04,0.22


In [15]:
df_indicadores_curso_ano.to_parquet("teste_indicadores_curso_ano.parquet", index=False)

In [16]:
df_indicadores_cor_curso_ano = calcular_indicadores(df, ["Ano", "Cor / Raça", "Nome de Curso"])
df_indicadores_cor_curso_ano.head(20)

,Ano,Cor / Raça,Nome de Curso,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes_no_prazo,concluintes_com_atraso,abandono,desligados,transferencia_interna,transferencia_externa,integralizadas,concluintes_previstos,taxa_conclusao,taxa_evasao,taxa_retencao,indice_eficiencia,taxa_matr_cont_regular,taxa_permanencia_exito,taxa_efetividade_academica
0,2017,Amarela,Gestão Desportiva e de Lazer,1,1,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
1,2017,Branca,Análise e Desenvolvimento de Sistemas,157,101,20,3,9,17,4,2,1,0,15,7.64,14.01,12.74,8.33,64.33,71.97,0.20
2,2017,Branca,Eletrônica Industrial,53,36,0,0,0,13,4,0,0,0,8,0.00,32.08,0.00,0.00,67.92,67.92,0.00
3,2017,Branca,Gestão Desportiva e de Lazer,62,40,4,5,2,8,2,0,0,1,13,12.90,16.13,6.45,29.41,64.52,77.42,0.38
4,2017,Branca,Letras - Português e Espanhol,18,15,0,0,0,0,3,0,0,0,0,0.00,16.67,0.00,0.00,83.33,83.33,0.00
5,2017,Branca,Técnico em Comércio,12,12,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
6,2017,Branca,Técnico em Eletrônica,67,43,2,7,2,0,6,6,1,0,15,13.43,10.45,2.99,31.82,64.18,77.61,0.47
7,2017,Branca,Técnico em Guia de Turismo,55,27,1,13,0,6,8,0,0,0,18,23.64,25.45,1.82,48.15,49.09,72.73,0.72
8,2017,Branca,Técnico em Informática,25,24,0,0,0,0,0,1,0,0,0,0.00,0.00,0.00,0.00,96.00,96.00,0.00
9,2017,Branca,Técnico em Lazer,22,21,0,0,0,0,0,1,0,0,0,0.00,0.00,0.00,0.00,95.45,95.45,0.00
